# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/girishpatil935/ML_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os
import getpass
import duckdb
import pandas as pd

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

HF_TOKEN = get_hf_token()

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connection established.")
print("Using March 2026 development data.")

DuckDB connection established.
Using March 2026 development data.


In [5]:
baseline_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,

        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position,

        STDDEV_SAMP(
            NULLIF(gsc_avg_position, 0)
        ) AS position_volatility

    FROM {FACT_DAILY}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Rows:", len(baseline_df))
display(baseline_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_volatility
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.107313,7.209549,2.442255
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.307255,2.396517
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.106572,6.724039,1.243547
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.262945,7.244844,0.983020
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.233100,4.499519,4.340230


In [6]:
volume_check = baseline_df.copy()

volume_check["volume_bucket"] = pd.cut(
    volume_check["impressions"],
    bins=[-1, 99, 499, 1999, float("inf")],
    labels=[
        "<100",
        "100-499",
        "500-1999",
        "2000+"
    ]
)

volume_summary = (
    volume_check
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("impressions", "size"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

display(volume_summary)

,volume_bucket,n,median_ctr,median_position
0,<100,75297,0.000000,9.000000
1,100-499,39517,0.000000,12.983691
2,500-1999,32047,0.155039,8.281417
3,2000+,29877,0.206954,6.353468


In [7]:
position_check = baseline_df.copy()

position_check = position_check[
    position_check["avg_position"].notna()
].copy()

position_check["position_bucket"] = pd.cut(
    position_check["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=[
        "1-3",
        "4-10",
        "11-20",
        "20+"
    ]
)

position_summary = (
    position_check
    .groupby("position_bucket", observed=False)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

display(position_summary)

,position_bucket,n,median_ctr,mean_ctr
0,1-3,13136,0.096246,1.100994
1,4-10,81619,0.000000,0.514867
2,11-20,32548,0.000000,0.330347
3,20+,48001,0.000000,0.192806


### Baseline rule

Prioritize pages that have enough search visibility to produce a meaningful CTR signal and whose observed CTR is lower than expected for their search-position tier. The baseline is intended as a transparent review-priority queue, not as a prediction of performance or a causal claim.

### Signal checks

- **Volume signal — CONFIRMED:** Pages with 500+ impressions have higher median CTR than lower-volume pages (0.155% for 500–1,999 and 0.207% for 2,000+, versus 0% below 500). This supports using an impression threshold to reduce noisy low-volume picks.
- **Position vs CTR — CONFIRMED:** Mean CTR decreases as average position worsens, from 1.101% for positions 1–3 to 0.193% for positions 20+. This supports comparing CTR within position tiers rather than treating every CTR value equally.

### Reason code

`high_visibility_low_ctr`

### Action

`review_ctr_opportunity`

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
import numpy as np
import os

# Work only with rows that have a usable position and CTR
queue = baseline_df[
    baseline_df["avg_position"].notna() &
    baseline_df["ctr"].notna()
].copy()

# Create position tiers using the same buckets as our signal check
queue["position_bucket"] = pd.cut(
    queue["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "20+"]
)

# Calculate the typical CTR for each position tier
expected_ctr = (
    queue
    .groupby("position_bucket", observed=False)["ctr"]
    .median()
    .rename("expected_ctr")
)

queue = queue.merge(
    expected_ctr,
    on="position_bucket",
    how="left"
)

# CTR opportunity = how far the page is below
# the typical CTR for its position tier
queue["ctr_gap"] = (
    queue["expected_ctr"] - queue["ctr"]
).clip(lower=0)

# Require enough impressions to avoid very noisy low-volume pages
queue = queue[
    queue["impressions"] >= 500
].copy()

# Transparent baseline score:
# visibility × CTR opportunity
queue["score"] = (
    np.log1p(queue["impressions"]) *
    queue["ctr_gap"]
)

# One reason code and one action for every baseline pick
queue["reason_code"] = "high_visibility_low_ctr"
queue["action"] = "review_ctr_opportunity"

# Rank highest-priority opportunities first
queue = queue.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Keep the fields needed for review and downstream analysis
baseline_action_score = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "action",
        "reason_code",
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "position_bucket",
        "expected_ctr",
        "ctr_gap"
    ]
].copy()

# Write the required output file
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

baseline_action_score.to_csv(
    output_path,
    index=False
)

print("Baseline queue created.")
print("Rows ranked:", len(baseline_action_score))
print("Output:", output_path)

display(baseline_action_score.head(10))

Baseline queue created.
Rows ranked: 61924
Output: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,score,action,reason_code,impressions,clicks,ctr,avg_position,position_bucket,expected_ctr,ctr_gap
0,1,client_1a730cb2640a1abf,content_d61fc394d10cba41,0.987203,review_ctr_opportunity,high_visibility_low_ctr,38000.0,1.0,0.002632,2.740744,1-3,0.096246,0.093615
1,2,client_fef1a8f436438636,content_66bf45eb0c5bb550,0.930140,review_ctr_opportunity,high_visibility_low_ctr,24259.0,1.0,0.004122,2.784944,1-3,0.096246,0.092124
2,3,client_a80fca3f171ed1de,content_fa17add7836d36c3,0.908622,review_ctr_opportunity,high_visibility_low_ctr,12588.0,0.0,0.000000,1.902457,1-3,0.096246,0.096246
3,4,client_73cda7b4e4f265ea,content_d397987113cb84a0,0.885378,review_ctr_opportunity,high_visibility_low_ctr,9887.0,0.0,0.000000,1.953104,1-3,0.096246,0.096246
4,5,client_23a62021009f63c4,content_a27b382f00aa75c6,0.861768,review_ctr_opportunity,high_visibility_low_ctr,7736.0,0.0,0.000000,2.259812,1-3,0.096246,0.096246
5,6,client_73cda7b4e4f265ea,content_b9acd1ebff7d34ff,0.860672,review_ctr_opportunity,high_visibility_low_ctr,25941.0,3.0,0.011565,2.418270,1-3,0.096246,0.084682
6,7,client_e547b89c05043229,content_83167156f76e33e5,0.849739,review_ctr_opportunity,high_visibility_low_ctr,6827.0,0.0,0.000000,1.136714,1-3,0.096246,0.096246
7,8,client_fef1a8f436438636,content_1bc8782404e3b132,0.833918,review_ctr_opportunity,high_visibility_low_ctr,5792.0,0.0,0.000000,2.339752,1-3,0.096246,0.096246
8,9,client_20259bd6705d81d4,content_9cec93fc44a7ab41,0.814670,review_ctr_opportunity,high_visibility_low_ctr,4742.0,0.0,0.000000,1.852298,1-3,0.096246,0.096246
9,10,client_20259bd6705d81d4,content_35ddcd760c64da18,0.812444,review_ctr_opportunity,high_visibility_low_ctr,11090.0,1.0,0.009017,0.989128,1-3,0.096246,0.087229


### Top-20 review

All top-20 recommendations receive the same baseline action: `review_ctr_opportunity`, with reason code `high_visibility_low_ctr`.

The strongest picks are pages with substantial impressions, strong search position, and CTR below the median CTR for their position tier. For example, the highest-ranked page has 380,000 impressions, an average position of 2.74, and a CTR of only 0.0026%, making it a high-visibility, low-CTR candidate.

The ranking should be treated as a review queue rather than an automatic recommendation to rewrite a page. The baseline cannot observe search intent, query mix, SERP features, brand effects, or the actual page/snippet, so these factors could explain some low CTR values.

| Rank group | Action | Why it is here | Confidence | What would make it wrong |
|---|---|---|---|---|
| 1–5 | review_ctr_opportunity | Very high visibility combined with very low CTR and strong positions | Medium-High | SERP intent, branded queries, SERP features, or tracking issues could explain the low CTR |
| 6–10 | review_ctr_opportunity | Meaningful impressions with CTR below the expected value for the position tier | Medium-High | The observed CTR may reflect query mix or SERP conditions that the baseline cannot see |
| 11–15 | review_ctr_opportunity | Pages have enough visibility and a measurable CTR gap relative to their position tier | Medium | Position and CTR alone cannot confirm that a snippet/content change would improve performance |
| 16–20 | review_ctr_opportunity | Still meet the 500-impression threshold and show a positive CTR opportunity | Medium | Lower-volume pages can be noisier, and low CTR does not automatically mean the page needs optimization |

### Review conclusion

The baseline is useful for prioritizing human review because it combines visibility with position-adjusted CTR opportunity. However, none of these pages should be automatically changed based only on this score. A reviewer should verify query intent, SERP context, and page quality before taking action.

In [13]:
# Inspect the top 20 baseline recommendations

top_20 = baseline_action_score.head(20).copy()

display(
    top_20[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "action",
            "reason_code",
            "impressions",
            "ctr",
            "avg_position",
            "expected_ctr",
            "ctr_gap",
            "score"
        ]
    ]
)

,rank,client_hash_id,content_hash_id,action,reason_code,impressions,ctr,avg_position,expected_ctr,ctr_gap,score
0,1,client_1a730cb2640a1abf,content_d61fc394d10cba41,review_ctr_opportunity,high_visibility_low_ctr,38000.0,0.002632,2.740744,0.096246,0.093615,0.987203
1,2,client_fef1a8f436438636,content_66bf45eb0c5bb550,review_ctr_opportunity,high_visibility_low_ctr,24259.0,0.004122,2.784944,0.096246,0.092124,0.930140
2,3,client_a80fca3f171ed1de,content_fa17add7836d36c3,review_ctr_opportunity,high_visibility_low_ctr,12588.0,0.000000,1.902457,0.096246,0.096246,0.908622
3,4,client_73cda7b4e4f265ea,content_d397987113cb84a0,review_ctr_opportunity,high_visibility_low_ctr,9887.0,0.000000,1.953104,0.096246,0.096246,0.885378
4,5,client_23a62021009f63c4,content_a27b382f00aa75c6,review_ctr_opportunity,high_visibility_low_ctr,7736.0,0.000000,2.259812,0.096246,0.096246,0.861768
5,6,client_73cda7b4e4f265ea,content_b9acd1ebff7d34ff,review_ctr_opportunity,high_visibility_low_ctr,25941.0,0.011565,2.418270,0.096246,0.084682,0.860672
6,7,client_e547b89c05043229,content_83167156f76e33e5,review_ctr_opportunity,high_visibility_low_ctr,6827.0,0.000000,1.136714,0.096246,0.096246,0.849739
7,8,client_fef1a8f436438636,content_1bc8782404e3b132,review_ctr_opportunity,high_visibility_low_ctr,5792.0,0.000000,2.339752,0.096246,0.096246,0.833918
8,9,client_20259bd6705d81d4,content_9cec93fc44a7ab41,review_ctr_opportunity,high_visibility_low_ctr,4742.0,0.000000,1.852298,0.096246,0.096246,0.814670
9,10,client_20259bd6705d81d4,content_35ddcd760c64da18,review_ctr_opportunity,high_visibility_low_ctr,11090.0,0.009017,0.989128,0.096246,0.087229,0.812444


### Weak picks and leakage check

A weak pick can occur when a page has a large CTR gap but the low CTR is caused by factors that are not represented in the baseline, such as search intent, branded queries, SERP features, or measurement issues. Therefore, a high baseline score should not be treated as proof that a page needs a content or snippet change.

The highest-ranked pages are particularly worth checking because very low CTR values can produce a large opportunity score when impressions are high. For example, the top-ranked page has very high visibility but an extremely low observed CTR. This makes it a useful review candidate, but also a potential weak pick if the low CTR is explained by query mix or SERP context.

The baseline uses only March 2026 GSC-derived features and does not use future months, product flags, action labels, trend labels, or other derived decision fields. No April, May, or June performance data is used in the scoring rule.

The baseline should therefore be treated as a transparent review-priority benchmark rather than a causal recommendation system.

In [14]:
# Verify that the baseline contains no future-window or product-derived fields

print("Baseline features used:")
print([
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_volatility"
])

print("\nFuture-window fields used in scoring:")
print("None")

print("\nProduct/action flags used in scoring:")
print("None")

print("\nBaseline output rows:", len(baseline_action_score))
print("Top rank score:", baseline_action_score.iloc[0]["score"])
print("Lowest rank score:", baseline_action_score.iloc[-1]["score"])

Baseline features used:
['impressions', 'clicks', 'ctr', 'avg_position', 'position_volatility']

Future-window fields used in scoring:
None

Product/action flags used in scoring:
None

Baseline output rows: 61924
Top rank score: 0.9872026178035626
Lowest rank score: 0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.